<a href="https://colab.research.google.com/github/pranavkantgaur/training_materials/blob/master/nuclear_reactor_lec_4_bsplines_multizone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 4: B-Splines for Multi-Zone Core Design
## Local Control and Complex Enrichment Patterns

### Objectives:
1. Review B-spline curves and local control property
2. Model multi-zone reactor cores with varying enrichments
3. Handle discontinuities at zone boundaries
4. Hands-on: Design 3-zone core with optimal enrichment profile

## B-Splines: Quick Review

### Definition
A **B-spline curve** is defined by:
- Control points: $P_0, P_1, \ldots, P_n$
- Degree: $p$ (typically 2 or 3)
- Knot vector: $U = [u_0, u_1, \ldots, u_m]$

### Parametric Form
$$C(u) = \sum_{i=0}^{n} N_{i,p}(u) P_i$$

Where $N_{i,p}(u)$ are **B-spline basis functions** defined recursively:

$$N_{i,0}(u) = \begin{cases} 1 & \text{if } u_i \leq u < u_{i+1} \\ 0 & \text{otherwise} \end{cases}$$

$$N_{i,p}(u) = \frac{u - u_i}{u_{i+p} - u_i}N_{i,p-1}(u) + \frac{u_{i+p+1} - u}{u_{i+p+1} - u_{i+1}}N_{i+1,p-1}(u)$$

### Key Properties
1. **Local control**: Moving control point $P_i$ affects curve only in range $[u_i, u_{i+p+1}]$
2. **Continuity**: $C^{p-k}$ continuous at knots of multiplicity $k$
3. **Convex hull**: Curve lies within convex hull of control points
4. **Flexibility**: Insert knots to add control without changing curve shape
5. **Partition of unity**: $\sum_i N_{i,p}(u) = 1$

### Why B-Splines for Multi-Zone Cores?
- **Local control**: Change one zone without affecting others
- **Discontinuities**: Repeated knots allow for jumps (zone boundaries)
- **Smooth within zones**: High-order continuity where desired
- **Physical mapping**: Control points ↔ fuel assembly enrichments

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import BSpline, splrep, splev
from scipy.optimize import minimize, differential_evolution
from scipy.integrate import odeint

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

In [ ]:
# B-spline basis function implementation
def bspline_basis(i, p, u, knots):
    """Cox-de Boor recursion for B-spline basis functions"""
    if p == 0:
        return 1.0 if knots[i] <= u < knots[i+1] else 0.0
    else:
        # Avoid division by zero
        denom1 = knots[i+p] - knots[i]
        term1 = (u - knots[i]) / denom1 * bspline_basis(i, p-1, u, knots) if denom1 != 0 else 0
        
        denom2 = knots[i+p+1] - knots[i+1]
        term2 = (knots[i+p+1] - u) / denom2 * bspline_basis(i+1, p-1, u, knots) if denom2 != 0 else 0
        
        return term1 + term2

def evaluate_bspline(u, control_points, degree, knots):
    """Evaluate B-spline curve at parameter u"""
    n = len(control_points) - 1
    result = 0.0
    for i in range(n + 1):
        result += bspline_basis(i, degree, u, knots) * control_points[i]
    return result

# Create example: uniform knot vector
def uniform_knot_vector(n, p):
    """Create uniform knot vector for n+1 control points and degree p"""
    m = n + p + 1
    knots = np.zeros(m + 1)
    # Clamped B-spline: repeat first and last knot p+1 times
    for i in range(p + 1):
        knots[i] = 0.0
        knots[m - i] = 1.0
    # Interior knots uniformly spaced
    for i in range(p + 1, m - p):
        knots[i] = (i - p) / (m - 2*p)
    return knots

# Visualize B-spline basis functions
n = 6  # 7 control points
p = 2  # quadratic
knots = uniform_knot_vector(n, p)
u_vals = np.linspace(0, 1, 500)

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
for i in range(n + 1):
    basis_vals = [bspline_basis(i, p, u, knots) for u in u_vals]
    plt.plot(u_vals, basis_vals, linewidth=2, label=f'N_{i},{p}(u)')
plt.xlabel('Parameter u', fontsize=12)
plt.ylabel('Basis Function Value', fontsize=12)
plt.title('B-Spline Basis Functions (p=2)', fontsize=14)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)

# Example curve
plt.subplot(1, 2, 2)
control_points = np.array([0, 0.5, 1.2, 0.8, 1.5, 1.0, 0.3])
curve_vals = [evaluate_bspline(u, control_points, p, knots) for u in u_vals]
plt.plot(u_vals, curve_vals, 'b-', linewidth=2.5, label='B-Spline Curve')
u_control = np.linspace(0, 1, n + 1)
plt.plot(u_control, control_points, 'ro-', markersize=8, alpha=0.5, label='Control Points')
plt.xlabel('Parameter u', fontsize=12)
plt.ylabel('Value', fontsize=12)
plt.title('Quadratic B-Spline Example', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Knot vector: {knots}")
print(f"Number of control points: {n + 1}")
print(f"Degree: {p}")

## Multi-Zone Core Design: Motivation

### Real Reactor Cores Have Multiple Zones:
1. **Fresh fuel** (high enrichment)
2. **Once-burned fuel** (medium enrichment)
3. **Twice-burned fuel** (lower enrichment)
4. **Control regions** (burnable poisons)
5. **Reflector zones** (no fuel)

### Design Challenges:
- **Discontinuities** at zone boundaries
- **Local adjustments** without global impact
- **Varied smoothness**: smooth within zones, jumps at boundaries
- **Physical constraints** per zone

### B-Spline Advantages:
- **Knot multiplicity**: Create discontinuities
- **Local support**: Change one zone independently
- **Knot insertion**: Refine specific regions
- **Natural mapping**: Each zone ↔ set of control points

## Connection to OpenMC Depletion Module

### OpenMC's Approach to Multi-Zone Depletion:

In OpenMC's `deplete` module:
- **Materials**: Each burnable material tracked independently
- **Operator**: Manages transport-depletion coupling
- **CRAM solver**: Uses Chebyshev Rational Approximation Method for Bateman equations
- **Reaction rates**: Computed via tallies for each material

### Our B-Spline Approach:
We use B-splines to:
1. **Represent enrichment profiles** across zones
2. **Interpolate material properties** between discrete assemblies
3. **Optimize zone boundaries** and enrichment values
4. **Smooth representation** of discrete zone structure

### Key Insight:
B-splines provide a **continuous representation** of the **discrete zone structure** that OpenMC uses, enabling:
- Gradient-based optimization
- Smooth perturbation analysis
- Design space exploration

In [ ]:
# Example 1: B-spline with zone boundaries using repeated knots

def create_multi_zone_knots(zone_boundaries, degree):
    """
    Create knot vector with repeated interior knots for discontinuities
    zone_boundaries: list of boundary locations in [0, 1]
    degree: B-spline degree
    """
    # Start with clamped ends
    knots = [0.0] * (degree + 1)
    
    # Add interior boundaries with multiplicity = degree (for C^0 continuity)
    for boundary in zone_boundaries:
        knots.extend([boundary] * degree)
    
    # End with clamped
    knots.extend([1.0] * (degree + 1))
    
    return np.array(knots)

# 3-zone reactor: boundaries at 1/3 and 2/3
zone_boundaries = [1/3, 2/3]
p = 2  # quadratic
knots_zones = create_multi_zone_knots(zone_boundaries, p)

# Number of control points
n_control = len(knots_zones) - p - 1

print(f"Knot vector with zone boundaries: {knots_zones}")
print(f"Number of control points: {n_control}")

# Define enrichment control points for 3 zones
# Zone 1 (fresh): high enrichment
# Zone 2 (once-burned): medium  
# Zone 3 (twice-burned): lower
control_enrichments = np.array([0.045, 0.045, 0.038, 0.038, 0.032, 0.032, 0.028])

# Evaluate enrichment profile
u_vals = np.linspace(0, 1, 1000)
enrichments = []
for u in u_vals:
    enr = evaluate_bspline(u, control_enrichments, p, knots_zones)
    enrichments.append(enr)
enrichments = np.array(enrichments)

# Map to position
L = 300  # reactor length (cm)
x_vals = u_vals * L

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Enrichment profile
axes[0].plot(x_vals, enrichments*100, 'b-', linewidth=2.5)
for boundary in zone_boundaries:
    axes[0].axvline(x=boundary*L, color='r', linestyle='--', linewidth=2, alpha=0.7)
axes[0].set_xlabel('Position x (cm)', fontsize=12)
axes[0].set_ylabel('Enrichment (%)', fontsize=12)
axes[0].set_title('3-Zone Enrichment Profile (B-Spline)', fontsize=14)
axes[0].grid(True, alpha=0.3)
axes[0].text(L/6, 4.6, 'Zone 1\n(Fresh)', ha='center', fontsize=11, 
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
axes[0].text(L/2, 3.9, 'Zone 2\n(Once-burned)', ha='center', fontsize=11,
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))
axes[0].text(5*L/6, 3.1, 'Zone 3\n(Twice-burned)', ha='center', fontsize=11,
            bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))

# Basis functions showing local support
u_fine = np.linspace(0, 1, 500)
for i in range(n_control):
    basis = [bspline_basis(i, p, u, knots_zones) for u in u_fine]
    axes[1].plot(u_fine*L, basis, linewidth=1.5, label=f'N_{i}', alpha=0.7)
axes[1].set_xlabel('Position x (cm)', fontsize=12)
axes[1].set_ylabel('Basis Function Value', fontsize=12)
axes[1].set_title('B-Spline Basis (Local Support)', fontsize=14)
axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
axes[1].grid(True, alpha=0.3)
for boundary in zone_boundaries:
    axes[1].axvline(x=boundary*L, color='r', linestyle='--', linewidth=1, alpha=0.5)

plt.tight_layout()
plt.show()

print(f"\nEnrichment in Zone 1: {np.mean(enrichments[:333])*100:.3f}%")
print(f"Enrichment in Zone 2: {np.mean(enrichments[333:666])*100:.3f}%")
print(f"Enrichment in Zone 3: {np.mean(enrichments[666:])*100:.3f}%")

## Demonstrating Local Control

**Key advantage of B-splines**: Changing one control point affects only nearby region.

This is crucial for reactor design:
- Adjust Zone 2 enrichment without changing Zones 1 or 3
- Insert new control point to add local refinement
- Optimize each zone independently

In [ ]:
# Demonstrate local control: perturb middle zone control points
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

perturbations = [0, 0.003, -0.003, 0.005]  # perturb Zone 2
perturb_indices = [3, 4]  # control points for Zone 2

for idx, (ax, pert) in enumerate(zip(axes.flat, perturbations)):
    control_pert = control_enrichments.copy()
    for i in perturb_indices:
        control_pert[i] += pert
    
    enrichments_pert = [evaluate_bspline(u, control_pert, p, knots_zones) 
                        for u in u_vals]
    
    ax.plot(x_vals, np.array(enrichments_pert)*100, 'b-', linewidth=2.5)
    ax.plot(x_vals, enrichments*100, 'k--', linewidth=1.5, alpha=0.5, label='Original')
    
    for boundary in zone_boundaries:
        ax.axvline(x=boundary*L, color='r', linestyle='--', linewidth=1, alpha=0.5)
    
    ax.set_xlabel('Position x (cm)', fontsize=11)
    ax.set_ylabel('Enrichment (%)', fontsize=11)
    ax.set_title(f'Perturbation = {pert:+.3f} in Zone 2', fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.set_ylim([2.5, 5])
    if idx == 0:
        ax.legend()

plt.tight_layout()
plt.show()

print("Notice: Perturbations to Zone 2 control points affect ONLY Zone 2!")
print("Zones 1 and 3 remain unchanged → LOCAL CONTROL")

## Example 2: Multi-Zone Flux Optimization

**Design Problem**: 
- 3-zone core with distinct enrichments
- Minimize peak-to-average flux ratio
- Maintain k_eff ≈ 1.0
- Enforce zone-specific enrichment bounds

We'll use the diffusion solver from Lecture 3 with B-spline enrichment profile.

In [ ]:
# Simplified 1D diffusion solver (from Lecture 3)
def solve_1d_diffusion_multizone(L, enrichment_func, n_points=500):
    """
    Solve 1D diffusion with position-dependent enrichment
    Returns: x, phi, k_eff, enrichments
    """
    x = np.linspace(0, L, n_points)
    dx = x[1] - x[0]
    
    # Material properties
    D = 1.0  # diffusion coefficient
    barn = 1e-24
    rho_U = 19.1  # g/cm^3
    N_A = 6.022e23
    A_U = 238
    N_total = rho_U * N_A / A_U * barn
    
    # Get enrichment at each position
    t_pos = x / L
    enrichments = np.array([enrichment_func(t) for t in t_pos])
    
    # Cross-sections
    sigma_f_U235 = 585 * barn
    sigma_a_U235 = 681 * barn
    sigma_a_U238 = 2.7 * barn
    nu = 2.43
    
    # Macroscopic cross-sections
    nu_Sigma_f = np.zeros(n_points)
    Sigma_a = np.zeros(n_points)
    
    for i in range(n_points):
        N_U235 = enrichments[i] * N_total
        N_U238 = (1 - enrichments[i]) * N_total
        nu_Sigma_f[i] = nu * sigma_f_U235 * N_U235
        Sigma_a[i] = sigma_a_U235 * N_U235 + sigma_a_U238 * N_U238
    
    # Build matrices for eigenvalue problem
    A = np.zeros((n_points, n_points))
    M = np.zeros((n_points, n_points))
    
    for i in range(1, n_points-1):
        A[i, i-1] = -D / dx**2
        A[i, i] = 2*D / dx**2 + Sigma_a[i]
        A[i, i+1] = -D / dx**2
        M[i, i] = nu_Sigma_f[i]
    
    # Boundary conditions
    A[0, 0] = 1.0
    A[-1, -1] = 1.0
    
    # Solve eigenvalue problem
    from scipy.linalg import eig
    eigenvalues, eigenvectors = eig(A, M)
    
    # Get largest real eigenvalue
    real_eigs = np.real(eigenvalues[np.isreal(eigenvalues)])
    if len(real_eigs) > 0:
        k_eff = np.max(real_eigs)
        idx = np.argmax(np.real(eigenvalues))
        phi = np.abs(np.real(eigenvectors[:, idx]))
        phi = phi / np.max(phi)
    else:
        k_eff = 1.0
        phi = np.zeros(n_points)
    
    return x, phi, k_eff, enrichments

# Test with uniform enrichment
def uniform_enr(t):
    return 0.04

x_test, phi_test, k_test, enr_test = solve_1d_diffusion_multizone(L, uniform_enr)
print(f"Test with uniform 4% enrichment: k_eff = {k_test:.6f}")

In [ ]:
# Optimize 3-zone core using B-splines

def objective_multizone(control_points_flat):
    """Objective function for multi-zone optimization"""
    # Bounds checking
    if np.any(control_points_flat < 0.025) or np.any(control_points_flat > 0.05):
        return 1e10
    
    # Create enrichment function from B-spline
    def enrichment_func(u):
        return evaluate_bspline(u, control_points_flat, p, knots_zones)
    
    try:
        x, phi, k_eff, enrichments = solve_1d_diffusion_multizone(L, enrichment_func)
        
        # Calculate peak-to-average
        phi_pos = phi[phi > 0.01]
        if len(phi_pos) == 0:
            return 1e10
        
        peak_to_avg = np.max(phi_pos) / np.mean(phi_pos)
        
        # Criticality penalty
        k_penalty = 1000 * (k_eff - 1.0)**2
        
        # Zone smoothness penalty (discourage wild variations)
        smoothness_penalty = 10 * np.sum(np.diff(control_points_flat)**2)
        
        return peak_to_avg + k_penalty + smoothness_penalty
    except:
        return 1e10

# Initial guess: slightly varied enrichments per zone
initial_control = np.array([0.043, 0.044, 0.040, 0.039, 0.036, 0.035, 0.032])

print("Optimizing multi-zone core...")
print("This may take 1-2 minutes...\n")

# Optimize
bounds = [(0.025, 0.05) for _ in range(len(initial_control))]
result = minimize(objective_multizone, initial_control, method='L-BFGS-B',
                  bounds=bounds, options={'maxiter': 100})

optimal_control = result.x

print(f"Optimization converged: {result.success}")
print(f"Optimal control points: {optimal_control*100}\n")

# Evaluate optimal solution
def optimal_enrichment(u):
    return evaluate_bspline(u, optimal_control, p, knots_zones)

x_opt, phi_opt, k_opt, enr_opt = solve_1d_diffusion_multizone(L, optimal_enrichment)
phi_opt_pos = phi_opt[phi_opt > 0.01]
pa_opt = np.max(phi_opt_pos) / np.mean(phi_opt_pos)

# Compare with uniform
print(f"Uniform enrichment:")
print(f"  k_eff: {k_test:.6f}")
phi_test_pos = phi_test[phi_test > 0.01]
pa_uni = np.max(phi_test_pos) / np.mean(phi_test_pos)
print(f"  Peak-to-average: {pa_uni:.4f}")

print(f"\nOptimized multi-zone:")
print(f"  k_eff: {k_opt:.6f}")
print(f"  Peak-to-average: {pa_opt:.4f}")
print(f"\nImprovement: {(pa_uni - pa_opt)/pa_uni*100:.1f}% reduction in P/A ratio")

In [ ]:
# Plot comprehensive results
fig = plt.figure(figsize=(15, 10))
gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)

# Flux profiles
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(x_test, phi_test, 'b-', linewidth=2, label='Uniform', alpha=0.7)
ax1.plot(x_opt, phi_opt, 'r-', linewidth=2.5, label='Optimized Multi-Zone')
ax1.axhline(y=np.mean(phi_opt_pos), color='green', linestyle='--', alpha=0.5, label='Average (opt)')
for boundary in zone_boundaries:
    ax1.axvline(x=boundary*L, color='purple', linestyle=':', linewidth=2, alpha=0.5)
ax1.set_xlabel('Position x (cm)', fontsize=12)
ax1.set_ylabel('Normalized Flux', fontsize=12)
ax1.set_title('Flux Profile Comparison', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Enrichment profiles
ax2 = fig.add_subplot(gs[1, 0])
ax2.plot(x_test, enr_test*100, 'b-', linewidth=2, label='Uniform', alpha=0.7)
ax2.plot(x_opt, enr_opt*100, 'r-', linewidth=2.5, label='Optimized')
u_ctrl = np.linspace(0, 1, len(optimal_control))
x_ctrl = u_ctrl * L
ax2.plot(x_ctrl, optimal_control*100, 'go', markersize=8, label='Control Points')
for boundary in zone_boundaries:
    ax2.axvline(x=boundary*L, color='purple', linestyle=':', linewidth=2, alpha=0.5)
ax2.set_xlabel('Position x (cm)', fontsize=11)
ax2.set_ylabel('Enrichment (%)', fontsize=11)
ax2.set_title('Enrichment Profiles', fontsize=13)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

# Power distribution
ax3 = fig.add_subplot(gs[1, 1])
ax3.fill_between(x_test, 0, phi_test, alpha=0.3, color='blue', label='Uniform')
ax3.fill_between(x_opt, 0, phi_opt, alpha=0.5, color='red', label='Optimized')
for boundary in zone_boundaries:
    ax3.axvline(x=boundary*L, color='purple', linestyle=':', linewidth=2, alpha=0.5)
ax3.set_xlabel('Position x (cm)', fontsize=11)
ax3.set_ylabel('Power Density', fontsize=11)
ax3.set_title('Power Distribution', fontsize=13)
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)

# Peak-to-average comparison
ax4 = fig.add_subplot(gs[2, 0])
cases = ['Uniform', 'Multi-Zone\nB-Spline']
pa_vals = [pa_uni, pa_opt]
colors = ['blue', 'red']
bars = ax4.bar(cases, pa_vals, color=colors, alpha=0.7, width=0.5)
ax4.set_ylabel('Peak-to-Average Ratio', fontsize=11)
ax4.set_title('Flux Flattening Performance', fontsize=13)
ax4.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, pa_vals):
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Zone statistics
ax5 = fig.add_subplot(gs[2, 1])
zone_starts = [0, int(n_points*1/3), int(n_points*2/3)]
zone_ends = [int(n_points*1/3), int(n_points*2/3), n_points]
zone_labels = ['Zone 1\n(Fresh)', 'Zone 2\n(Once-burned)', 'Zone 3\n(Twice-burned)']
zone_enr_avg = []
zone_flux_avg = []

for start, end in zip(zone_starts, zone_ends):
    zone_enr_avg.append(np.mean(enr_opt[start:end]) * 100)
    zone_flux_avg.append(np.mean(phi_opt[start:end]))

x_pos = np.arange(len(zone_labels))
ax5_twin = ax5.twinx()
bar1 = ax5.bar(x_pos - 0.2, zone_enr_avg, 0.4, label='Enrichment', color='orange', alpha=0.7)
bar2 = ax5_twin.bar(x_pos + 0.2, zone_flux_avg, 0.4, label='Flux', color='cyan', alpha=0.7)
ax5.set_ylabel('Average Enrichment (%)', fontsize=11, color='orange')
ax5_twin.set_ylabel('Average Flux', fontsize=11, color='cyan')
ax5.set_xlabel('Zone', fontsize=11)
ax5.set_title('Zone-wise Statistics', fontsize=13)
ax5.set_xticks(x_pos)
ax5.set_xticklabels(zone_labels, fontsize=9)
ax5.tick_params(axis='y', labelcolor='orange')
ax5_twin.tick_params(axis='y', labelcolor='cyan')
ax5.grid(True, alpha=0.3, axis='y')

plt.suptitle('Multi-Zone Core Optimization Results', fontsize=16, fontweight='bold', y=0.995)
plt.show()

# Print zone statistics
print("\n=== Zone-wise Analysis ===")
for i, label in enumerate(zone_labels):
    print(f"{label}:")
    print(f"  Average enrichment: {zone_enr_avg[i]:.3f}%")
    print(f"  Average flux: {zone_flux_avg[i]:.4f}")
    print()

## Summary

### What We Learned:
1. ✅ **B-splines** provide local control through basis function support
2. ✅ **Multi-zone cores** modeled with repeated knots for discontinuities
3. ✅ **Local optimization** possible without affecting other zones
4. ✅ **Flux flattening** achieved across multiple enrichment zones
5. ✅ **Connection to OpenMC**: B-splines as continuous representation of discrete zones

### Key Results:
- Significant improvement in peak-to-average ratio
- Each zone can be optimized independently
- Smooth profiles within zones, discontinuities at boundaries
- Realistic representation of actual reactor loading patterns

### Advantages of B-Splines:
- **Local control**: Best for multi-zone problems
- **Flexibility**: Easily add/remove zones (knots)
- **Smoothness control**: Vary continuity as needed
- **Computational efficiency**: Sparse basis functions

### Comparison with Previous Methods:
| Method | Global/Local | Discontinuities | Best For |
|--------|-------------|-----------------|----------|
| Hermite | Local (segment) | Yes (at joints) | Time evolution |
| Bezier | Global | No | Simple geometries |
| B-spline | Local | Yes (knot multiplicity) | **Multi-zone cores** |

**Next Lecture**: We'll explore **tally derivatives** and flux sensitivity analysis using curve derivatives to accelerate the transport-depletion loop!

## Exercises

1. **Knot Insertion**:
   - Start with 2 zones
   - Insert knots to create a 4-zone core
   - How does this affect the optimization?

2. **Variable Degree**:
   - Try cubic B-splines (p=3) instead of quadratic
   - How does increased degree affect smoothness?
   - Trade-off: more control points vs smoother curves

3. **Reflector Regions**:
   - Add reflector zones (no fuel) at boundaries
   - Set enrichment = 0 in these regions
   - How does this change the flux shape?

4. **Time-Dependent Multi-Zone**:
   - Combine with Lecture 2: depletion in each zone
   - Use B-splines for spatial profile at each time step
   - How do zones burn at different rates?

5. **Burnable Poisons**:
   - Add a 4th zone with burnable poison (high absorption)
   - Design to control excess reactivity
   - Optimize poison concentration and location